# HIV-1 *pol* subtype classification — per-class analysis

Deep-dive on the three models the paper centres on, all evaluated on the **same
leakage-free, cluster-grouped test split** (seed 0), so the numbers line up with
the main multi-seed run:

- **LinearSVM** on TF-IDF 6-mer — the simple baseline the deep models must beat
- **ResNet-18 (1D)** on TF-IDF 6-mer — best CNN overall
- **VGG-16 (1D)** on raw 6-mer frequency — co-leader, *different* encoding

**Outputs** (to `/kaggle/working/`):
- `per_class_metrics.csv` — precision, recall, F1, support, AUROC, AUPRC per class + a MACRO row, for every model
- `confusion_matrix_linearsvm.csv`, `confusion_matrix_resnet18.csv`, `confusion_matrix_vgg16.csv`

Runtime on one T4 ≈ 40–50 min (SVM ~2 min, ResNet ~15–20 min, VGG ~20–25 min) — well inside the session limit.

> Accelerator: **GPU T4 ×2**. Commit with *Save & Run All* to run headless.

### 1. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, glob, random
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, average_precision_score)
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, GaussianNoise,
    SpatialDropout1D, Layer)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("TF", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

### 2. Config

In [ ]:
SEED = 0            # same seed as the main run -> the split matches the reported metrics
OUT  = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

### 3. Load data

Looks for `prepared_sequences_clean.csv` anywhere under `/kaggle/input` or `/kaggle/working`.
If only the un-cleaned `prepared_sequences.csv` is present, it rebuilds the clean 10-class
set on the fly (same KEEP list as Script A cleanup).

In [ ]:
def find_file(name):
    for base in ["/kaggle/input", "/kaggle/working", "."]:
        hits = glob.glob(f"{base}/**/{name}", recursive=True)
        if hits: return hits[0]
    return None

path = find_file("prepared_sequences_clean.csv")
if path is not None:
    df = pd.read_csv(path)
    print("loaded clean set:", path)
else:
    raw = find_file("prepared_sequences.csv")
    if raw is None:
        raise FileNotFoundError(
            "Add prepared_sequences_clean.csv (or prepared_sequences.csv) as a Kaggle dataset input.")
    df = pd.read_csv(raw)
    KEEP = ["B","C","A1","A6","D","G","01_AE","02_AG","07_BC","08_BC"]   # 6 subtypes + 4 CRFs
    df = df[df["subtype"].isin(KEEP)].copy()
    df.to_csv(f"{OUT}/prepared_sequences_clean.csv", index=False)
    print("rebuilt clean set from:", raw)

le = LabelEncoder(); y = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)       # fixed class order used everywhere below
X = df["sequence"].values; grp = df["cluster"].values
print(f"{len(df)} seqs | {n_cls} classes: {CLASSES}")
print(df["subtype"].value_counts().to_string())

### 4. Leakage-free split (grouped by cluster, stratified by class)

`StratifiedGroupKFold` keeps every cluster wholly inside one split, so near-duplicate
sequences can't leak across train/test, while balancing class proportions.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trv, te = next(sgkf.split(X, y, grp))                     # fold 0 test = 20%
sgkf2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(sgkf2.split(X[trv], y[trv], grp[trv]))
tr, va = trv[tr_rel], trv[va_rel]
X_tr, X_va, X_te = X[tr], X[va], X[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)))
print(f"train={len(tr)} val={len(va)} test={len(te)}")

### 5. Encoders (fitted on TRAIN only)

Two feature sets, because the two CNNs won with different encodings:
- **TF-IDF 6-mer** → LinearSVM and ResNet-18
- **raw 6-mer frequency** (length-normalised) → VGG-16

In [ ]:
def kmers(seq, k=6): return [seq[i:i+k] for i in range(len(seq)-k+1)]

# TF-IDF 6-mer: vocabulary + idf weights learned on TRAIN only
vec = TfidfVectorizer(max_features=512)
vec.fit([" ".join(kmers(s)) for s in X_tr])
tfidf = lambda seqs: vec.transform([" ".join(kmers(s)) for s in seqs]).toarray().astype(np.float32)

# Raw 6-mer frequency: vocabulary from TRAIN only, each row divided by its length
def make_kmer_encoder(train, k=6):
    vocab = sorted({km for s in train for km in kmers(s, k)}); idx = {kk:i for i,kk in enumerate(vocab)}
    def enc(seqs):
        out = np.zeros((len(seqs), len(vocab)), np.float32)
        for i, s in enumerate(seqs):
            for km in kmers(s, k):
                if km in idx: out[i, idx[km]] += 1
            t = out[i].sum()
            if t: out[i] /= t
        return out
    return enc
kfreq = make_kmer_encoder(X_tr, 6)

Xtr_t, Xva_t, Xte_t = tfidf(X_tr), tfidf(X_va), tfidf(X_te)     # TF-IDF features
Xtr_k, Xva_k, Xte_k = kfreq(X_tr), kfreq(X_va), kfreq(X_te)     # raw 6-mer features
print("TF-IDF dim:", Xtr_t.shape[1], "| 6-mer dim:", Xtr_k.shape[1])

### 6. Per-class metric helper

For one model, builds a table of per-class precision / recall / F1 / support plus
per-class one-vs-rest AUROC and AUPRC, appends a MACRO (unweighted mean) row, and
saves that model's confusion matrix. `y_score` is the per-class continuous score
(softmax probs for CNNs, `decision_function` margins for the SVM) — both AUROC and
AUPRC accept scores, not just probabilities.

In [ ]:
def per_class_table(y_true, y_pred, y_score, model_name):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(n_cls), zero_division=0)
    y_oh = np.eye(n_cls)[y_true]                          # one-hot truth for OVR curves
    aur, aup = [], []
    for c in range(n_cls):
        pos = y_oh[:, c].sum()
        if pos == 0 or pos == len(y_true):                # metric undefined if class absent from test
            aur.append(np.nan); aup.append(np.nan)
        else:
            aur.append(roc_auc_score(y_oh[:, c], y_score[:, c]))
            aup.append(average_precision_score(y_oh[:, c], y_score[:, c]))
    t = pd.DataFrame({"model": model_name, "class": CLASSES, "support": s,
                      "precision": np.round(p,4), "recall": np.round(r,4), "f1": np.round(f,4),
                      "auroc": np.round(aur,4), "auprc": np.round(aup,4)})
    macro = {"model": model_name, "class": "MACRO", "support": int(s.sum()),
             "precision": round(p.mean(),4), "recall": round(r.mean(),4), "f1": round(f.mean(),4),
             "auroc": round(np.nanmean(aur),4), "auprc": round(np.nanmean(aup),4)}
    t = pd.concat([t, pd.DataFrame([macro])], ignore_index=True)
    cm = confusion_matrix(y_true, y_pred, labels=range(n_cls))   # rows = true, cols = predicted
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(f"{OUT}/confusion_matrix_{model_name}.csv")
    return t

### 7. Baseline — LinearSVM (TF-IDF 6-mer)

In [ ]:
svm = LinearSVC(class_weight="balanced")
svm.fit(Xtr_t, y_tr)
svm_scores = svm.decision_function(Xte_t)                 # (n, n_cls) margins -> OVR scores
tbl_svm = per_class_table(y_te, svm.predict(Xte_t), svm_scores, "linearsvm")
print(tbl_svm.to_string(index=False))

### 8. ResNet-18 (1D) and VGG-16 (1D) architectures

In [ ]:
# ---- ResNet-18 (1D) ----
class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1]); return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(L, n):
    inp = Input((L, 1)); x = GaussianNoise(0.05)(inp)
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for ff, sp in [(32,.95),(32,.92)]:   x = res_block(x, ff, 1, sp)
    for ff, sp in [(64,.88),(64,.85)]:   x = res_block(x, ff, 2 if ff!=x.shape[-1] else 1, sp)
    for ff, sp in [(128,.80),(128,.78)]: x = res_block(x, ff, 2 if ff!=x.shape[-1] else 1, sp)
    for ff, sp in [(256,.75),(256,.72)]: x = res_block(x, ff, 2 if ff!=x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    m = Model(inp, Dense(n, activation="softmax")(x))
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

# ---- VGG-16 (1D): classic 13-conv stack, filters scaled for 1D ----
def build_vgg16(L, n):
    def cb(x, f, k):
        for _ in range(k):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((L, 1))
    x = cb(inp, 32, 2); x = cb(x, 64, 2); x = cb(x, 128, 3); x = cb(x, 256, 3); x = cb(x, 256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    m = Model(inp, Dense(n, activation="softmax")(x))
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

### 9. Train helper

Scales features (fit on train only), trains a fresh model with the same callbacks as
the main run, clears the previous graph so the GPU doesn't fill up, and returns the
per-class table. No models are saved to disk.

In [ ]:
def train_cnn(build_fn, Xtr_f, Xva_f, Xte_f, name):
    sc = StandardScaler()
    a = sc.fit_transform(Xtr_f)[..., None]                # scaler fit on TRAIN only
    b = sc.transform(Xva_f)[..., None]
    c = sc.transform(Xte_f)[..., None]
    tf.keras.backend.clear_session()
    m = build_fn(a.shape[1], n_cls)
    m.fit(a, to_categorical(y_tr, n_cls), validation_data=(b, to_categorical(y_va, n_cls)),
          epochs=120, batch_size=48, class_weight=cw, shuffle=True, verbose=0,
          callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                     ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = m.predict(c, verbose=0)
    return per_class_table(y_te, prob.argmax(1), prob, name)

### 10. ResNet-18 on TF-IDF 6-mer

In [ ]:
tbl_resnet = train_cnn(build_resnet18, Xtr_t, Xva_t, Xte_t, "resnet18")
print(tbl_resnet.to_string(index=False))

### 11. VGG-16 on raw 6-mer

In [ ]:
tbl_vgg = train_cnn(build_vgg16, Xtr_k, Xva_k, Xte_k, "vgg16")
print(tbl_vgg.to_string(index=False))

### 12. Save + show everything

In [ ]:
out = pd.concat([tbl_svm, tbl_resnet, tbl_vgg], ignore_index=True)
out.to_csv(f"{OUT}/per_class_metrics.csv", index=False)
pd.set_option("display.width", 200)
print("=== per-class metrics (all models) ===")
print(out.to_string(index=False))
print("\nSaved to /kaggle/working/:")
for fn in ["per_class_metrics.csv", "confusion_matrix_linearsvm.csv",
           "confusion_matrix_resnet18.csv", "confusion_matrix_vgg16.csv"]:
    print("  •", fn)